In [ ]:
import glob
import os
import numpy as np
import pandas as pd
import scienceplots
import seaborn as sns
import yaml
from matplotlib import pyplot as plt
from tqdm import tqdm
from scipy import stats
from matplotlib.ticker import PercentFormatter
import marsilea as ma
import marsilea.plotter as mp
from pybedtools import BedTool
plt.style.use(["science", "nature"])
import pickle

plt.rcParams['xtick.labelsize'] = 5
plt.rcParams['ytick.labelsize'] = 5
plt.rcParams['axes.labelsize'] = 6
plt.rcParams["xtick.top"] = False
plt.rcParams["ytick.right"] = False
plt.rcParams["lines.linewidth"] = 0.5
plt.rcParams["legend.fontsize"] = 6
plt.rcParams['hatch.linewidth'] = 0.5

# %matplotlib widget

protocol_map = {
    "Visium": "10X Visium",
    "VisiumHD": "10X Visium HD",
    "Chromium": "10X Chromium",
    "Dropseq": "Drop-seq",
    "Stereoseq": "Stereo-seq",
    "Slideseq": "Slide-seq V2",
    "SpatialTranscriptomics": "ST",
    "Microwell": "Microwell-seq",
    "annotation": "Annotated PAS",
    "Annotation": "Annotated PAS",
    "anno": "Annotated PAS",
}
type_map = {
    "Visium": "Spatial transcriptome",
    "VisiumHD": "Spatial transcriptome",
    "Chromium": "scRNA-seq",
    "Dropseq": "scRNA-seq",
    "Stereoseq": "Spatial transcriptome",
    "Slideseq": "Spatial transcriptome",
    "SpatialTranscriptomics": "Spatial transcriptome",
    "Microwell": "scRNA-seq",
}
order = ["10X Chromium", "Drop-seq", "Microwell-seq", "10X Visium","Stereo-seq", "Slide-seq V2", "ST"]
# order = ["10X Chromium", "Drop-seq", "Microwell-seq", "10X Visium", "10X Visium HD","Stereo-seq", "Slide-seq V2", "Spatial Transcriptomics"]

color = [
    "#386b98",
    "#269a51",
    "#edaa4d",
    "#d34123",
    "#7e648a",
    "#454545",
    "#929292",
]
palette=sns.color_palette(color, 7)
mm = 1/25.4

In [ ]:
import json
import glob
import pandas as pd
from collections import defaultdict

def process_json_files(model_result_path):
    """处理JSON结果文件并生成结构化DataFrame"""
    
    # 初始化数据容器
    data_dict = defaultdict(list)
    sample_ids = []
    
    # 遍历所有JSON文件
    for idx, json_file in enumerate(glob.glob(model_result_path + "*.json")):
        with open(json_file) as f:
            data = json.load(f)
            
        # 生成样本ID（可根据需要自定义）
        sample_id = "_".join(os.path.basename(json_file).split('.')[0].split("_")[0:4])
        sample_ids.append(sample_id)
        
        data_size = int(data["data_size"])
        apex_mean = float(data["apex_mean"])
        apex_std = float(data["apex_std"])
        weighted_mean = float(data["weighted_mean"])
        weighted_std = float(data["weighted_std"])


        # 提取自定义模型数据
        for prefix in ["custom_", "normal_"]:
            model_type = "custom_model" if prefix == "custom_" else "normal_dist"
            
            # 参数提取
            params = data[model_type]["params"]
            for param, value in params.items():
                data_dict[f"{prefix}{param}"].append(value)
                
            # 指标提取
            for metric in ["log_likelihood", "AIC", "BIC"]:
                data_dict[f"{prefix}{metric}"].append(data[model_type][metric])
        
        data_dict["data_size"].append(data_size)
        data_dict["apex_mean"].append(apex_mean)
        data_dict["apex_std"].append(apex_std)
        data_dict["weighted_mean"].append(weighted_mean)
        data_dict["weighted_std"].append(weighted_std)

    # 转换为DataFrame
    df = pd.DataFrame(data_dict)
    df.insert(0, "sample_id", sample_ids)  # 添加样本ID列
    
    # 列排序（可选）
    column_order = ["sample_id"] 
    column_order += [f"custom_{x}" for x in ["mu1","sigma1","sigma2","a","b","log_likelihood","AIC","BIC"]]
    column_order += [f"normal_{x}" for x in ["mu","sigma","log_likelihood","AIC","BIC"]]
    column_order += ["data_size","apex_mean","apex_std","weighted_mean","weighted_std"]
    
    return df[column_order]

# 使用示例
model_result_path = "../../data/int_data/model_result/"
combined_df = process_json_files(model_result_path)



In [ ]:
combined_df["protocol"] = combined_df["sample_id"].str.split("_").str[0]
combined_df["tissue"] = combined_df["sample_id"].str.split("_").str[2]
combined_df["species"] = combined_df["sample_id"].str.split("_").str[1]
combined_df["delta_AIC_custom"] = combined_df["custom_AIC"] - combined_df[["custom_AIC", "normal_AIC"]].min(axis=1)
combined_df["delta_AIC_normal"] = combined_df["normal_AIC"] - combined_df[["custom_AIC", "normal_AIC"]].min(axis=1)

In [ ]:
combined_df["delta_AIC"] = combined_df["custom_AIC"] - combined_df["normal_AIC"]
combined_df["delta_BIC"] = combined_df["custom_BIC"] - combined_df["normal_BIC"]

In [ ]:
plt.figure(figsize=(30*mm, 30*mm))
sns.boxplot(data=combined_df[['delta_AIC', 'delta_BIC']], palette=palette)
# plt.title('Comparison of AIC and BIC Distributions')
plt.tick_params(axis='x', which='minor', length=0)
plt.axhline(0, color='gray', linestyle='--')  # Add reference line at zero
plt.savefig('../../figures/AIC_BIC_boxplot.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
combined_df["protocol"] = combined_df["sample_id"].str.split("_").str[0]
combined_df["protocol"] = combined_df["protocol"].map(protocol_map)
combined_df["protocol"] = pd.Categorical(combined_df["protocol"], categories=order)
combined_df = combined_df.sort_values("protocol")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 需要可视化的列
parameters = ['custom_mu1', 'custom_sigma1', 'custom_sigma2', 'custom_a', 'custom_b']
params_map = {
    'custom_mu1': 'μ1',
    'custom_sigma1': 'σ1',
    'custom_sigma2': 'σ2',
    'custom_a': 'a',
    'custom_b': 'b'
}

# 设置画布和子图布局（1行5列）
fig, axes = plt.subplots(1, len(parameters), figsize=(150*mm, 30*mm), sharey=True)

# 生成箱线图
for i, (ax, param) in enumerate(zip(axes, parameters)):
    sns.boxplot(
        data=combined_df[combined_df["custom_a"] > -1000],
        x=param,       # 用protocol作为分组维度
        y='protocol',  # 用protocol作为分组维度
        hue='protocol',  # 用protocol作为分组维度
        # orient='h',         # 横向箱线图
        ax=ax,               # 指定当前子图
        palette=palette,
        legend=False
    )
    # ax.set_title(param)     # 设置子图标题
    # ax.set_xlabel('Value')  # 设置x轴标签
    
    # # 仅最左侧子图显示y轴标签
    ax.set_ylabel('')
    if i != 0:
        # set y tick invisible
        ax.tick_params(axis='y', which='both', length=0)
    else:
        ax.tick_params(axis='y', which='minor', length=0)
    ax.margins(y=0.06,x=0.07)


# 调整子图间距
plt.subplots_adjust(wspace=0.04)
plt.savefig('../../figures/model_params.pdf', bbox_inches='tight', dpi=300)
# plt.tight_layout()
plt.show()
